In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import sys
import os
project_path=os.path.join(os.getcwd(),'..','..')
sys.path.append(project_path)
from UTILS.MyClass import Vivo_Project


In [0]:
print(os.getcwd())

In [0]:
df= spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "parquet") \
  .option("cloudFiles.schemaLocation", "abfss://silver@vivostorageaccountname.dfs.core.windows.net/DimUser/checkpoint") \
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
  .load("abfss://bronze@vivostorageaccountname.dfs.core.windows.net/output/DimUser")

In [0]:
display(df, checkpointLocation="abfss://silver@vivostorageaccountname.dfs.core.windows.net/DimUser/checkpoint")

In [0]:
df= df.withColumn("user_name", upper(col("user_name")))

In [0]:
df= df.drop("_rescued_data")

In [0]:
df.writeStream.format("delta").outputMode("append").option("checkpointLocation", "abfss://silver@vivostorageaccountname.dfs.core.windows.net/DimUser/schema").trigger(once=True).toTable("vivo_project.silver.DimUser")

In [0]:
df1= spark.table("vivo_project.silver.DimUser")
df1.display()

## FactStream

In [0]:
dffs= spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "parquet") \
  .option("cloudFiles.schemaLocation", "abfss://silver@vivostorageaccountname.dfs.core.windows.net/FactStream/Schema") \
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
  .load("abfss://bronze@vivostorageaccountname.dfs.core.windows.net/output/FactStream")

In [0]:
dffs= dffs.drop("_rescued_data")

In [0]:
dffs.writeStream.format("delta").outputMode("append").option("checkpointLocation", "abfss://silver@vivostorageaccountname.dfs.core.windows.net/FactStream/checkpoint").trigger(once=True).toTable("vivo_project.silver.FactStream")

In [0]:
abc= spark.table("vivo_project.silver.FactStream")
abc.display()

## DimTrack

In [0]:
dfdt= spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "parquet") \
  .option("cloudFiles.schemaLocation", "abfss://silver@vivostorageaccountname.dfs.core.windows.net/DimTrack/Schema") \
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
  .load("abfss://bronze@vivostorageaccountname.dfs.core.windows.net/output/DimTrack")

In [0]:
dfdt= dfdt.drop("_rescued_data")

In [0]:
dfdt.writeStream.format("delta").outputMode("append").option("checkpointLocation", "abfss://silver@vivostorageaccountname.dfs.core.windows.net/DimTrack/checkpoint").trigger(once=True).toTable("vivo_project.silver.DimTrack")

In [0]:
ab= spark.table("vivo_project.silver.DimTrack")
ab.display()

In [0]:
obj= Vivo_Project()
obj.callme(ab)